# MCP Server

[![Open In Colab - incident agent runner](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/proloy79/incident_agent/blob/main/notebooks/incident_agent_runner.ipynb)

In [ ]:
from pathlib import Path
import os 
import sys

def in_colab():
    return 'google.colab' in sys.modules

if in_colab():
    # --- Colab‑only setup ---
    print("Running in Colab — setting up environment")
    if not os.path.exists('/content/incident_agent'):
      !git clone https://github.com/proloy79/incident_agent.git
    !pip install -e /content/incident_agent
    sys.path.append('/content/incident_agent/src')
    
    !pip install cloudflared
    !apt-get update
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !mv cloudflared-linux-amd64 cloudflared
    !chmod +x cloudflared
    !./cloudflared --version
    
else:
    raise("Notebook setup not done for local use yet!")

## Start server

In [ ]:
import subprocess
import re

server = subprocess.Popen(
    ["python", "-m", "incident_agent/mcp_demo_server"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("MCP demo server started.")

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8765"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Extract the public URL from cloudflared output
public_url = None
for line in tunnel.stdout:
    print(line, end="")
    match = re.search(r"https://.*?trycloudflare.com", line)
    if match:
        public_url = match.group(0)
        break

print("\nPublic MCP endpoint:", public_url)

FileNotFoundError: [Errno 2] No such file or directory: 'cloudflared'